# 6. Inference

Reads all embedding files from R2, runs the trained MLP model and writes one parquet per embedding file with the keyword and 7 labels:
- `intent` — string: informational / commercial / transactional / navigational
- `entity_type` — string: person / event / location / company / product / other
- `is_adult` — float (regression score)
- `saas_potential` — float (regression score)
- `is_question` — int (0/1)
- `is_comparison` — int (0/1)
- `is_branded` — int (0/1)

Outputs are saved to `keywordsdb/inference/` on R2, mirroring the embedding file names.

Steps:
1. Add `service_account.json` secret with a service account with access to R2.
2. Run training notebook (5_training) first so the MLP model exists.
3. **Enable GPU accelerator** (2x T4).
4. Save and commit so it runs in the background.

In [ ]:
import json
import multiprocessing as mp
import sys

import s3fs  # ty:ignore[unresolved-import]
from kaggle_secrets import UserSecretsClient  # ty:ignore[unresolved-import]

user_secrets = UserSecretsClient()
r2_creds = json.loads(user_secrets.get_secret("service_account.json"))
print("Authenticated with R2 access key:", r2_creds.get("access_key_id"))

R2_EMBEDDINGS = "keywordsdb/embeddings/raw"
R2_MODELS = "keywordsdb/models"
R2_INFERENCE = "keywordsdb/inference"

# Binary classification thresholds — update from training notebook output if available
# (all_results["mlp"][task]["threshold"])
BINARY_THRESHOLDS: dict[str, float] = {"is_question": 0.64, "is_comparison": 0.11, "is_branded": 0.49}

FS = s3fs.S3FileSystem(
    key=r2_creds["access_key_id"], secret=r2_creds["secret_access_key"], endpoint_url=r2_creds["endpoint_url"]
)

In [ ]:
embedding_files = sorted(f.split("/")[-1] for f in FS.glob(f"{R2_EMBEDDINGS}/*.npz"))
already_done = {f.split("/")[-1].replace(".parquet", ".npz") for f in FS.glob(f"{R2_INFERENCE}/*.parquet")}

pending = [f for f in embedding_files if f not in already_done]
print(f"Total embedding files : {len(embedding_files)}")
print(f"Already processed     : {len(already_done)}")
print(f"Pending               : {len(pending)}")

In [ ]:
%%writefile /kaggle/working/_inference_worker.py
import io
import os
import queue
import re
import threading
import time

import numpy as np
import polars as pl
import s3fs


INTENT_LABELS = ["informational", "commercial", "transactional", "navigational"]
ENTITY_LABELS = ["person", "event", "location", "company", "product", "other"]
EMBEDDING_DIM = 384
INPUT_DIM = EMBEDDING_DIM + 5

QUESTION_WORDS = re.compile(
    r"^(how|what|why|where|when|which|who|whom|whose|is|are|can|could|does|do|will|should)\b", re.IGNORECASE
)
VS_PATTERN = re.compile(r"\b(vs\.?|versus|compared to|comparison)\b", re.IGNORECASE)


def run_on_gpu(
    gpu_id: int,
    file_names: list,
    r2_creds: dict,
    r2_embeddings: str,
    r2_models: str,
    r2_inference: str,
    binary_thresholds: dict,
) -> None:
    # Pin this process to one GPU before any CUDA/PyTorch imports
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    import torch
    import torch.nn as nn

    device = "cuda" if torch.cuda.is_available() else "cpu"

    fs = s3fs.S3FileSystem(
        key=r2_creds["access_key_id"],
        secret=r2_creds["secret_access_key"],
        endpoint_url=r2_creds["endpoint_url"],
    )

    # --- Model definition (must match 5_training) ---
    task_configs = [
        ("intent", "multiclass", len(INTENT_LABELS)),
        ("entity_type", "multiclass", len(ENTITY_LABELS)),
        ("is_adult", "regression", 1),
        ("saas_potential", "regression", 1),
        ("is_question", "binary", 1),
        ("is_comparison", "binary", 1),
        ("is_branded", "binary", 1),
    ]

    class MultiTaskMLP(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.backbone = nn.Sequential(
                nn.Linear(INPUT_DIM, 1024),
                nn.LayerNorm(1024),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(1024, 512),
                nn.LayerNorm(512),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(512, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.1),
            )
            self.heads = nn.ModuleDict({
                task_key: nn.Linear(256, n_out)
                for task_key, _, n_out in task_configs
            })

        def forward(self, x: torch.Tensor) -> dict:
            h = self.backbone(x)
            return {task_key: self.heads[task_key](h) for task_key in self.heads}

    # --- Load model ---
    model = MultiTaskMLP().to(device)
    with fs.open(f"{r2_models}/mlp/model.pt", "rb") as f:
        state_dict = torch.load(io.BytesIO(f.read()), map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"[GPU{gpu_id}] Loaded MLP model", flush=True)

    # --- Helpers ---
    def _text_features(keywords: list) -> np.ndarray:
        feats = np.zeros((len(keywords), 5), dtype=np.float32)
        for i, kw in enumerate(keywords):
            feats[i, 0] = len(kw)
            feats[i, 1] = len(kw.split())
            feats[i, 2] = float(bool(QUESTION_WORDS.search(kw)))
            feats[i, 3] = float(bool(VS_PATTERN.search(kw)))
            feats[i, 4] = float(bool(re.search(r"\d", kw)))
        return feats

    def _run_inference(keywords: list, embeddings: np.ndarray) -> pl.DataFrame:
        text_feats = _text_features(keywords)
        x = np.hstack([embeddings, text_feats])
        x_t = torch.from_numpy(x).float().to(device)
        with torch.no_grad():
            raw = model(x_t)
        result: dict = {"keyword": keywords}
        if "intent" in raw:
            result["intent"] = [INTENT_LABELS[i] for i in raw["intent"].cpu().argmax(dim=1).tolist()]
        if "entity_type" in raw:
            result["entity_type"] = [ENTITY_LABELS[i] for i in raw["entity_type"].cpu().argmax(dim=1).tolist()]
        if "is_adult" in raw:
            result["is_adult"] = raw["is_adult"].cpu().squeeze(-1).tolist()
        if "saas_potential" in raw:
            result["saas_potential"] = raw["saas_potential"].cpu().squeeze(-1).tolist()
        for flag in ["is_question", "is_comparison", "is_branded"]:
            if flag in raw:
                probs = torch.sigmoid(raw[flag].cpu().squeeze(-1)).numpy()
                threshold = binary_thresholds.get(flag, 0.5)
                result[flag] = (probs > threshold).astype(np.int8).tolist()
        return pl.DataFrame(result)

    def _prefetch_worker(names, out_q):
        for name in names:
            with fs.open(f"{r2_embeddings}/{name}", "rb") as f:
                data = np.load(io.BytesIO(f.read()), allow_pickle=False)
                keywords = data["keywords"].astype(str).tolist()
                embeddings = data["embeddings"].astype(np.float32)
            out_q.put((name, keywords, embeddings))
        out_q.put(None)

    def _write_worker(write_q):
        while True:
            item = write_q.get()
            if item is None:
                break
            name, df = item
            out_path = f"{r2_inference}/{name.replace('.npz', '.parquet')}"
            buf = io.BytesIO()
            df.write_parquet(buf)
            buf.seek(0)
            with fs.open(out_path, "wb") as f:
                f.write(buf.read())
            print(f"[GPU{gpu_id}] Saved: {name.replace('.npz', '.parquet')}, rows={len(df)}", flush=True)

    # --- Main loop with prefetch + async write ---
    prefetch_q: queue.Queue = queue.Queue(maxsize=8)
    write_q: queue.Queue = queue.Queue(maxsize=4)
    prefetch_t = threading.Thread(target=_prefetch_worker, args=(file_names, prefetch_q), daemon=True)
    write_t = threading.Thread(target=_write_worker, args=(write_q,), daemon=True)
    prefetch_t.start()
    write_t.start()

    files_done = 0
    t_start = time.perf_counter()
    while True:
        item = prefetch_q.get()
        if item is None:
            break
        name, keywords, embeddings = item
        t_file = time.perf_counter()
        df = _run_inference(keywords, embeddings)
        write_q.put((name, df))
        files_done += 1
        elapsed_file = time.perf_counter() - t_file
        elapsed_total = time.perf_counter() - t_start
        print(
            f"[GPU{gpu_id}] Inferred: {name}, progress={files_done}/{len(file_names)}, "
            f"file={elapsed_file:.1f}s, total={elapsed_total:.0f}s",
            flush=True,
        )

    write_q.put(None)
    prefetch_t.join()
    write_t.join()
    print(f"[GPU{gpu_id}] Done: {files_done} files in {time.perf_counter() - t_start:.0f}s", flush=True)

In [ ]:
sys.path.insert(0, "/kaggle/working")
from _inference_worker import run_on_gpu  # ty:ignore[unresolved-import]

mp.set_start_method("spawn", force=True)

mid = len(pending) // 2
splits = [pending[:mid], pending[mid:]]
print(f"Splitting {len(pending)} files across 2 GPUs: {len(splits[0])} / {len(splits[1])}")

processes = [
    mp.Process(
        target=run_on_gpu, args=(gpu_id, subset, r2_creds, R2_EMBEDDINGS, R2_MODELS, R2_INFERENCE, BINARY_THRESHOLDS)
    )
    for gpu_id, subset in enumerate(splits)
]

for p in processes:
    p.start()
for p in processes:
    p.join()

print("All GPUs done.")